In [10]:
import polars as pl
import orjsonl
from collections import defaultdict

In [ ]:
main_path = "../data/out/分子筛/20260108160000/main.jsonl"
char_paths = [
    "../data/out/分子筛/20260108160000/char_20260109135019.jsonl",
    "../data/out/分子筛/20260108160000/char.jsonl",
]
app_paths = [
    "../data/out/分子筛/20260108160000/app_20260109135019.json",
    "../data/out/分子筛/20260108160000/app.jsonl"
]
syn_paths = [
    "../data/out/分子筛/20260108160000/syn_20260109134837.jsonl",
    "../data/out/分子筛/20260108160000/syn.jsonl",
]
outpath = r"../data/out/分子筛/20260108160000/merged.jsonl"

In [12]:
# 先进行各个步骤的合并
# materail_id
results = defaultdict(dict)
for record in orjsonl.stream(main_path):
    if record["main_extract_result"]["error"]:
        continue
        
    material_ids = record["main_extract_result"]["parse"]["material_ids"] 
    
    for mid in material_ids:
        id_index = (record["file_name"], mid["id"])
        results[id_index]["doi"] = record["doi"]
        results[id_index]["aliases"] = mid["aliases"]
        results[id_index]["framework_code"] = mid["framework_code"]
for cp in char_paths:
    for record in orjsonl.stream(cp):
        if record["syn_extract_result"]["error"] is not None:
            continue
        else:
            results[(record["file_name"], record["material_id"])]["characterization"] = record["syn_extract_result"]["parse"]["characterization"]
for cp in app_paths:
    for record in orjsonl.stream(cp):
        if record["syn_extract_result"]["error"] is not None:
            continue
        results[(record["file_name"], record["material_id"])]["application"] = record["syn_extract_result"]["parse"]["application"]

for cp in syn_paths:
    for record in orjsonl.stream(cp):
        if record["syn_extract_result"]["error"] is not None:
            continue
        results[(record["file_name"], record["material_id"])]["synthesis"] = record["syn_extract_result"]["parse"]["synthesis"]


FileNotFoundError: [Errno 2] No such file or directory: '../data/out/分子筛/20260108160000/char_20260109135019.jsonl../data/out/分子筛/20260108160000/char.jsonl'

In [ ]:
import orjson
out_records = []
for k,v in results.items():
    framework_code = v.get("framework_code")
    if framework_code is None:
        framework_code = v["synthesis"].get("zeolite_type")
    
    synthesis = v.get("synthesis", {})
    synthesis["zeolite_type"] = framework_code
    
    out_records.append({
        "file_name": k[0],
        "doi": v.get("doi"),
        
        "material_id": k[1],
        
        "aliases": v.get("aliases"),
        "synthesis": v.get("synthesis"),
        "characterization": v.get("characterization"),
        "application": v.get("application"),
    })
out_records = sorted(out_records, key = lambda r:(r["file_name"], r["material_id"]))

In [15]:
orjsonl.save(outpath, out_records)

In [10]:
len(results.keys())

160

In [89]:
len(results.keys()),results.keys()

(608,
 dict_keys([('1 (2).md', 'HCGPE'), ('1 (2).md', 'HCE'), ('1-s2.0-S0009250925009364-main.md', 'LAWTP'), ('1-s2.0-S0009250925009364-main.md', 'PVDF-HFP-SN'), ('1-s2.0-S0009250925009364-main.md', 'PVDF-HFP-SN-0.05LAWTP'), ('1-s2.0-S0009250925009364-main.md', 'PVDF-HFP-SN-0.1LAWTP'), ('1-s2.0-S0009250925009364-main.md', 'PVDF-HFP-SN-0.2LAWTP'), ('1-s2.0-S0013468620312561-main.md', '[HMG][FSI]'), ('1-s2.0-S0013468623016389-main.md', '(PMMA)0.91(IM14)0.09'), ('1-s2.0-S0013468623016389-main.md', '(PMMA)0.91(TFSI)0.09'), ('1-s2.0-S0013468623016389-main.md', '(PMMA)0.91(TFO)0.09'), ('1-s2.0-S0013468623016389-main.md', '(DBUH-IM14)0.60-(PMMA)0.40'), ('1-s2.0-S0013468623016389-main.md', '(DBUH-IM14)0.55(Li)0.06-(PMMA)0.39'), ('1-s2.0-S0014305705000649-main.md', 'M1'), ('1-s2.0-S0014305705000649-main.md', 'M2'), ('1-s2.0-S0014305705000649-main.md', 'PM1'), ('1-s2.0-S0014305705000649-main.md', 'PM2'), ('1-s2.0-S0032386116306772-main.md', 'PFPE_E10-DMC'), ('1-s2.0-S0167273817305489-main.md', '

In [90]:
results[('1 (2).md', 'HCGPE')]

{'doi': '10.1002/adfm.202412287',
 'aliases': [],
 'framework_code': None,
 'characterization': {'morphology': 'amorphous',
  'crystal_size': None,
  'si_al_ratio_actual': None,
  'porosity': {'bet_m2_g': None,
   'v_total_cm3_g': None,
   'v_micro_cm3_g': None,
   'v_meso_cm3_g': None},
  'acidity': {'bronsted_acid_amount_mmol_g': None,
   'lewis_acid_amount_mmol_g': None,
   'b_l_ratio': None}},
 'application': {'scenario': 'battery_electrolyte',
  'target_species': 'Li+',
  'capacity': None,
  'selectivity': None,
  'breakthrough': None,
  'regeneration': None},
 'synthesis': {'zeolite_type': None,
  'gel_composition': 'HCGPE precursor: 5 wt.% TFEMA + 95 wt.% HCE (LiFSI:DME = 1.9 molar ratio); crosslinker PETA 10 wt.% vs TFEMA; initiator AIBN 1 wt.% vs TFEMA',
  'template': None,
  'silica_source': None,
  'aluminium_source': None,
  'crystallization': {'temp_c': 60.0, 'time_d': 0.5, 'agitation_rpm': None},
  'post_treatment_steps': []}}